In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import csv, re
import pandas as pd

Languages are matched against a hand-written list of language names, independent of any KB, so
`languages` does not depend on ESCO or ONET. Job descriptions almost never state the modality
(spoken, written, ...), so we keep only the language and the raw snippet for later refinement.

In [ ]:
# Hand-written, KB-independent. Dead languages (Latin, Sanskrit, ...) are left out: "Latin"
# would also match "Latin America".
LANGUAGES = [
    "Albanian", "Arabic", "Armenian", "Basque", "Bengali", "Bosnian", "Bulgarian", "Catalan",
    "Chinese", "Croatian", "Czech", "Danish", "Dutch", "English", "Estonian", "Finnish",
    "French", "Galician", "Georgian", "German", "Greek", "Hebrew", "Hindi", "Hungarian",
    "Icelandic", "Irish", "Italian", "Japanese", "Korean", "Latvian", "Lithuanian",
    "Luxembourgish", "Macedonian", "Malay", "Maltese", "Norwegian", "Persian", "Polish",
    "Portuguese", "Punjabi", "Romanian", "Russian", "Serbian", "Slovak", "Slovenian",
    "Spanish", "Swedish", "Turkish", "Ukrainian", "Urdu", "Vietnamese", "Welsh",
]

In [ ]:
# Descriptions are already machine-translated to English (1_translate_job_title), so the
# language names can be matched verbatim.
jobs = pd.read_csv("mapping/translated.csv")[["id_j", "eng_description"]]
jobs["eng_description"] = jobs["eng_description"].fillna("")
jobs.head()

In [ ]:
LANGUAGE_RE = re.compile(r"\b(" + "|".join(map(re.escape, LANGUAGES)) + r")\b")

# A bare language name is usually not a requirement ("Spanish market", "Latin America",
# "El Corte Ingles"). Only keep mentions sitting next to a language-requirement cue.
CUE_RE = re.compile(
    r"\b(language|fluen\w*|nativ\w*|bilingual|proficien\w*|spoken|speak\w*|written|"
    r"write|read\w*|command|level|[ABC][12])\b",
    re.IGNORECASE,
)
WINDOW = 80

def extract_languages(description: str):
    """Return one record per (language, cue-supported mention) found in the description."""
    found = {}
    for m in LANGUAGE_RE.finditer(description):
        context = description[max(0, m.start() - WINDOW): m.end() + WINDOW]
        if CUE_RE.search(context) and m.group(1) not in found:
            found[m.group(1)] = context.strip()
    return found

assert extract_languages("Native English speaker required").keys() == {"English"}
assert extract_languages("We serve the Spanish market from Latin America") == {}
assert extract_languages("B2 English and some French language skills").keys() == {"English", "French"}

In [ ]:
OUTPUT_FILE = "mapping/language_map.csv"
DETAILED_FILE = "mapping/language_map_detailed.csv"

aggregate_rows, detailed_rows = [], []
for row in jobs.itertuples(index=False):
    found = extract_languages(row.eng_description)
    aggregate_rows.append({
        "id_j": row.id_j,
        "languages": sorted(found),
    })
    detailed_rows += [
        {"id_j": row.id_j, "language": lang, "context": context}
        for lang, context in sorted(found.items())
    ]

language_map = pd.DataFrame(aggregate_rows)
language_detailed = pd.DataFrame(detailed_rows)

language_map.to_csv(OUTPUT_FILE, quoting=csv.QUOTE_ALL, index=False)
language_detailed.to_csv(DETAILED_FILE, quoting=csv.QUOTE_ALL, index=False)

print(f"{(language_map.languages.str.len() > 0).sum()} / {len(language_map)} offers with a language requirement")

In [ ]:
language_detailed.language.value_counts().head(15)